<!-- HEADER-KLAIM -->
# 06 — Kalau Bukan ANN, Apakah XGBoost Lebih Baik?

| | |
|---|---|
| **Pertanyaan** | Akurasi 17 kelas mentok. Apakah penyebabnya classifier-nya (ANN), atau fiturnya? |
| **Cara membuktikan** | Data, window, dan 17 skenario **identik** dengan notebook `01`. Yang diganti hanya bagian bawah: evaluasi per-skenario, penambahan fitur time-domain, dan classifier XGBoost sebagai pembanding ANN. |
| **Gunanya** | Kalau XGBoost pun mentok di angka yang mirip, berarti batasnya ada di **fitur/tugas**, bukan di ANN — ini yang memperkuat argumen bahwa tugas 17 kelas memang tumpang-tindih. |
| **Status** | Eksperimen pendamping, bukan hasil utama paper. |

---


# Notebook 3/3 - Revisi: Per-Skenario + Fitur Time-Domain + XGBoost

**Eksperimen yang diuji:** deteksi fault pada 4 sensor kelembaban, sama persis datanya dengan 2
notebook lain (`Rencana_Paper_JSD_Fuzzy_Q3.ipynb`, `final_databaru_jsd_improved.ipynb`) - fault
injection, windowing, dan 17 skenario fault IDENTIK (cell 0-33 di bawah adalah pipeline yang sama).
Revisinya ada di bagian bawah notebook (setelah cell fitur entropy):

1. **Klasifikasi per-skenario** (S1-S5) - tiap skenario dilatih + grid search TERPISAH, bukan 1
   model untuk 17 kelas sekaligus.
2. **Fitur time-domain** (mean, std, RMS, skew, kurtosis, zero-crossing, slope, energy, FFT 3-band)
   ditambahkan sebagai fitur baru di luar entropy, dihitung dari sinyal mentah `W_s`.
3. **Classifier XGBoost** menggantikan MLP.
4. **Ablasi feature-set**: tiap skenario dicoba 5 kombinasi fitur (EDM, JSD, Time, EDM+Time,
   JSD+Time) untuk melihat kontribusi masing-masing.

**Beda dengan 2 notebook lain di repo ini:**
- Fault injection & data loading: SAMA PERSIS (lihat markdown "# Fault Injection (per Sensor)" di
  bawah - simulator fault, `inject_faults_multisensor`, windowing, semua identik dengan Rencana/
  final_databaru).
- Yang beda: classifier (MLP -> XGBoost), unit klasifikasi (17-kelas sekaligus -> per-skenario
  S1-S5 terpisah), dan fitur tambahan (entropy saja -> entropy + time-domain).
- Hasil: F1-macro rata-rata naik dari ~0.57 (entropy+MLP, 17-kelas) ke ~0.78 (JSD+Time+XGBoost,
  per-skenario), runtime turun dari ~7 jam ke ~37 menit.

Pipeline sesuai diagram: baseline -> fault injection per sensor -> hitung EDM-Fuzzy Entropy per
sensor (vector per skala) -> gabung fitur -> [revisi] + fitur time-domain -> XGBoost per-skenario.

**Catatan percepatan**: notebook ini memakai (1) downsampling & window sampling, (2) perkiraan
(Monte Carlo) pada perhitungan similarity untuk menekan kompleksitas $O(N^2)$.

# Evaluasi CV utuh untuk panjang data berbeda + uji formula hidden layer (P1–P7)

Bagian ini menambahkan dua eksperimen:

## A) CV utuh pada ukuran data berbeda (contoh: 2000, 7000, 10000)
Untuk setiap ukuran `N`, ambil subsample acak dari `(X_feat, y)`, lalu jalankan **5-fold Stratified CV penuh**.

## B) Uji formula jumlah neuron hidden layer (P1–P7)
Konversi formula P1–P7 menjadi angka neuron `h`, lalu evaluasi `MLPClassifier(hidden_layer_sizes=(h,))` dengan **CV penuh** dan metrik `f1_macro`.

> Catatan: hasil CV akan tidak stabil jika jumlah window sangat sedikit atau label sangat tidak seimbang.


In [ ]:
import numpy as np, math
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

def eval_for_sizes(X, y, sizes=(2000,7000,10000), scoring="f1_macro", cv_splits=5, seed=42):
    rng=np.random.default_rng(seed)
    idx_all=np.arange(len(y))
    out={}
    for n in sizes:
        k=min(int(n), len(y))
        idx=rng.choice(idx_all, size=k, replace=False)
        Xn, yn = X[idx], y[idx]
        cv=StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
        pipe=Pipeline([("scaler", StandardScaler()),
                       ("clf", MLPClassifier(hidden_layer_sizes=(128,64), activation="tanh",
                                             max_iter=400, early_stopping=True, random_state=seed))])
        scores=cross_val_score(pipe, Xn, yn, cv=cv, scoring=scoring, n_jobs=1)
        out[n]={"mean":float(scores.mean()),"std":float(scores.std()),"scores":scores}
        print(f"N={k:5d} | mean={scores.mean():.4f} std={scores.std():.4f} | scores={np.round(scores,4)}")
    return out

def hl_formulas(I, O, Nt):
    return {
        "P1": int(2*I + 1),
        "P2": max(1, int(math.log2(max(2,I)))),
        "P3": max(1, int((I + O) / 2)),
        "P4": max(1, int((2/3) * I)),
        "P5": int(2 * I),
        "P6": max(1, int(I / 2 + 1)),
        "P7": max(1, int(0.5 * (I + O) + math.sqrt(max(1,Nt))))
    }

def eval_formulas(X, y, formulas, scoring="f1_macro", cv_splits=5, seed=42, activation="tanh"):
    cv=StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    rows=[]
    for name,h in formulas.items():
        pipe=Pipeline([("scaler", StandardScaler()),
                       ("clf", MLPClassifier(hidden_layer_sizes=(h,), activation=activation,
                                             max_iter=400, early_stopping=True, random_state=seed))])
        scores=cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=1)
        rows.append((name,h,float(scores.mean()),float(scores.std())))
        print(f"{name}: HL={h:4d} | mean={scores.mean():.4f} std={scores.std():.4f}")
    rows=sorted(rows, key=lambda t: t[2], reverse=True)
    return rows


In [ ]:
# === Global config (edit here) ===
FAST_MODE = True  # True: fastest path to get all outputs
RUN_ALL_METHODS = True  # True: compute + evaluate all entropy methods
METHOD_LIST = ["EDM-Fuzzy", "JSD-Fuzzy"]
DEFAULT_METHOD = "EDM-Fuzzy"  # used for plots/CM/report if you want 1 method highlighted
CACHE_FEATURES = True  # cache features per method to avoid recompute on rerun
CACHE_DIR = "cache"
EXPORT_DIR = "exports"

# FAST_MODE knobs (will override some later defaults)
FAST_MAX_PER_CLASS = 200
FAST_N_REF = 128
FAST_N_JOBS = -1
FAST_MLP_MAX_ITER = 200
FAST_CV_REPEATS = 10  # for entropy stability repeats (if used)

# Kaggle time budget guard (hours)
KAGGLE_TIME_BUDGET_H = 11.5

# Per-scenario ANN search knobs (paper table)
SCENARIO_GRID_CV = 3
SCENARIO_TEST_FRAC = 0.25
SCENARIO_MAX_PER_CLASS = 300
SCENARIO_MAX_CANDIDATES = 10
SCENARIO_MAX_ITER = 250

import os
from pathlib import Path
from IPython.display import FileLink, display

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)

def export_df(df, name, index=False):
    p_csv = Path(EXPORT_DIR) / f"{name}.csv"
    p_par = Path(EXPORT_DIR) / f"{name}.parquet"
    df.to_csv(p_csv, index=index)
    try:
        df.to_parquet(p_par, index=index)
    except Exception:
        p_par = None
    display(FileLink(str(p_csv)))
    if p_par is not None:
        display(FileLink(str(p_par)))
    return str(p_csv), (str(p_par) if p_par is not None else None)


## A) Jalankan CV untuk ukuran data berbeda
Jika `len(y) < 10000`, otomatis pakai ukuran maksimum yang tersedia.


In [ ]:
# Jalankan cell ini SETELAH X_feat dan y terbentuk.
if "X_feat" not in globals() or "y" not in globals():
    print("X_feat/y belum ada. Jalankan dulu bagian ekstraksi fitur (compute entropy -> X_feat) dan label y.")
else:
    import numpy as np
    print("X_feat shape:", X_feat.shape, " y:", y.shape, "classes:", np.unique(y))
    _ = eval_for_sizes(X_feat, y, sizes=(2000,7000,10000))


## B) Jalankan uji formula hidden layer (P1–P7)
`I` = jumlah fitur (`X_feat.shape[1]`)  
`O` = jumlah kelas (`len(unique(y))`)  
`Nt` = jumlah sampel (`len(y)`)


In [ ]:
# Jalankan cell ini SETELAH X_feat dan y terbentuk.
if "X_feat" not in globals() or "y" not in globals():
    print("X_feat/y belum ada. Jalankan dulu bagian ekstraksi fitur (compute entropy -> X_feat) dan label y.")
else:
    import numpy as np
    I = int(X_feat.shape[1])
    O = int(len(np.unique(y)))
    Nt = int(len(y))
    formulas = hl_formulas(I,O,Nt)
    print("I,O,Nt:", I,O,Nt)
    print("Formulas:", formulas)
    rows = eval_formulas(X_feat, y, formulas)
    print("\nRanked:")
    for name,h,mean,std in rows:
        print(f"{name}: HL={h:4d} | mean={mean:.4f} std={std:.4f}")


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import time, tracemalloc, logging, warnings
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# === Speed / sampling toggles (Kaggle-friendly defaults) ===
# Target: cepat di Kaggle CPU (2-4 core) tanpa meledakkan jumlah window/fitur.
USE_DOWNSAMPLE = True
USE_WINDOWING = True
USE_BALANCED_SUBSAMPLE = True
USE_RANDOM_WINDOW_SAMPLE = False  # True kalau masih terlalu lambat

# Downsample: ambil tiap DS sampel
DS = 4

# Windowing: banyak sampel, tapi ukuran window dibuat kecil agar entropy cepat
WIN = 256
STRIDE = 128

# Batasi dataset supaya entropy + gridsearch tidak lama
MAX_PER_CLASS = 200          # max window per kelas setelah label jadi
MAX_WINDOWS_TOTAL = 2000     # dipakai kalau USE_RANDOM_WINDOW_SAMPLE=True

RANDOM_SEED = 42


# Load Data (4 Sensor)

**Tujuan:** memuat time-series kelembaban, memilih **4 kolom numerik** sebagai sensor, lalu membersihkan nilai hilang.

**Input**
- File CSV `tabel_sensor4_generated.csv` (diunduh via URL GitHub).
- Terdapat kolom "kelembaban1","kelembaban2","kelembaban3","kelembaban4"

**Proses di kode**
- `load_default_data()` mengunduh CSV → `pd.read_csv(...)` → DataFrame `df`.
- `cols = ["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]`
- `X_raw = df[cols].to_numpy(dtype=float)`
- Imputasi NaN berurutan: `ffill → bfill → median` per kolom.

**Output**
- `cols`: daftar nama 4 kolom sensor yang dipakai.
- `X`: `numpy.ndarray` bentuk `(T, 4)` berisi sinyal sensor yang sudah bebas NaN.


## 1) Load data (4 sensor)
File: `tabel_sensor4_generated.csv` (diasumsikan berisi 4 kolom sensor kelembaban).

In [ ]:
import pandas as pd
import requests

file_path = "tabel_sensor4_generated.csv"

def load_default_data():
    url = "https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/tabel_sensor4_generated.csv"
    response = requests.get(url)
    response.raise_for_status()
    from io import StringIO
    return pd.read_csv(StringIO(response.text))


df = load_default_data()
df.head(), df.shape


In [ ]:
# pilih 4 kolom
cols = ["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]
X_raw = df[cols].to_numpy(dtype=float)

# imputasi NaN sederhana (ffill → bfill → median)
X_df = pd.DataFrame(X_raw, columns=cols)
X_df = X_df.ffill().bfill().fillna(X_df.median(numeric_only=True))

# guard NaN (jelas kalau masih ada)
if X_df.isna().any().any():
    raise ValueError("Error-nya jelas: fitur X masih ada NaN setelah imputasi. Periksa data input.")

X = X_df.to_numpy()

# optional downsample (Kaggle speed)
if 'USE_DOWNSAMPLE' in globals() and USE_DOWNSAMPLE:
    X_ds = X[::DS]
else:
    X_ds = X

print("Columns:", cols)
print("Shape X:", X.shape, "Shape X_ds:", X_ds.shape)


In [ ]:
# --- Window params (pakai yg sudah didefinisikan di atas) ---
if "WIN" not in globals(): WIN = 256
if "STRIDE" not in globals(): STRIDE = max(1, WIN//2)
# Windowing parameters (FIX: aman walau X_ds belum didefinisikan)
# Catatan: cell ini sebaiknya berada SETELAH pembuatan X_ds. Kalau belum ada, kita fallback ke X.
import numpy as np


if "X_ds" not in globals():
    if "X" in globals():
        X_ds = X
    else:
        raise NameError("X_ds belum ada dan X belum ada. Jalankan dulu cell load/preprocess data.")

N = X_ds.shape[0]
if WIN > N:
    WIN = max(128, 2**int(np.floor(np.log2(max(128, N//2)))))
STRIDE = min(STRIDE, max(1, WIN//2))
print("WIN/STRIDE:", WIN, STRIDE, "N:", N)


In [ ]:
# Sanity check: jumlah window harus cukup banyak untuk klasifikasi
total_windows = None
if 'W' in globals(): total_windows = getattr(W, "shape", [None])[0]
if total_windows is not None and total_windows < 200:
    print("WARNING: total_windows terlalu sedikit:", total_windows)
    print("Saran cepat: coba WIN=512 STRIDE=128 atau WIN=1024 STRIDE=256 (atau turunkan STRIDE).")


# Fault Injection (per Sensor)

**Tujuan:** membangkitkan data *faulty* dari baseline dengan menyisipkan fault **per sensor** sesuai skenario.

## A. Simulator fault 1D (per sensor)
Semua simulator menerima **sinyal 1D** `x` bentuk `(T',)` dan menghasilkan:
- `y`: sinyal setelah fault `(T',)`
- `m`: mask boolean lokasi fault `(T',)`

- `simulate_drift_fault`: menambah drift linier `t*intensity`.
- `simulate_spike_fault`: menambah spike periodik (besar spike ∝ `std(x)`).
- `simulate_bias_fault`: menambah offset konstan (bias).
- `simulate_hardware_fault`: kombinasi **stuck** (nilai diganti nilai lain) dan **loss** (NaN).

## B. Multi-fault dalam satu sensor
`simulate_multiple_faults(x, faults)` menerapkan daftar fault berurutan; mask digabung `OR` sehingga menandai titik yang terkena salah satu fault.

## C. Injeksi ke 4 sensor
`inject_faults_multisensor(X, scenario_faults)`:
- loop ke-4 sensor: injeksi skenario yang sama, tetapi seed berbeda per sensor
- menangani NaN (loss) dengan `ffill → bfill → median`

**Output:**  
- `Y`: `(T',4)` data setelah fault  
- `M`: `(T',4)` mask fault per sensor

## D. Label fault per window
`window_fault_label(M, WIN, STRIDE, thr)`:
- untuk tiap window: hitung rasio fault per sensor = `mean(mask_window)`
- window dianggap fault bila **ada sensor** dengan rasio > `thr`

**Output:** `is_fault_win` boolean per window.

> **Catatan konsistensi:** blok fault-injection di atas (A-D) identik byte-for-byte dengan
> `Rencana_Paper_JSD_Fuzzy_Q3.ipynb` dan `final_databaru_jsd_improved.ipynb` - notebook ini
> memakai alur fault-detection yang SAMA, hanya bagian klasifikasi (di bawah) yang direvisi.

## 3) Fault injection (sesuai skenario)
Fault disisipkan **per sensor**. Skenario minimal 6 kombinasi 2-fault.

Output: dataset windows berlabel (kelas) + mask fault untuk menentukan window yang 'faulty'.

In [ ]:

# --- Fault simulators (subtler, different seed per call) ---
def simulate_drift_fault(x, intensity=0.02, seed=None):
    rng=np.random.default_rng(seed)
    alpha = intensity
    drift = np.arange(len(x)) * alpha
    y=x+drift; m=np.abs(drift)>1e-6; return y,m

def simulate_spike_fault(x, intensity=0.08, p=0.015, seed=None):
    rng=np.random.default_rng(seed)
    tau = max(1, int(1.0 / p)) if p > 0 else len(x)
    spikes = (np.arange(len(x)) % tau == 0).astype(float) * (intensity * np.nanstd(x))
    y=x+spikes; m=spikes!=0; return y,m

def simulate_bias_fault(x, bias=0.08, seed=None):
    y=x+bias; m=np.ones(len(x),bool); return y,m

def simulate_hardware_fault(x, stuck_prob=0.08, loss_prob=0.05, seed=None):
    rng=np.random.default_rng(seed)
    n = len(x)
    rand_vals = rng.random(n)
    idx=rng.integers(n, size=n)
    m1 = rand_vals < stuck_prob
    y = x.copy()
    y[m1] = x[idx[m1]]
    m2 = rand_vals < loss_prob
    y[m2] = np.nan
    return y, (m1 | m2)

def simulate_multiple_faults(x, faults, seed=None):
    y=x.copy(); m=np.zeros(len(x),bool)
    for f,kw in faults:
        y,mi=f(y,**kw,seed=seed); m|=mi
    return y,m


def simulate_choose_one(x, options, seed=None):
    rng = np.random.default_rng(seed)
    f, kw = options[rng.integers(len(options))]
    return f(x, **kw, seed=seed)

# fault scenario ditambah 2,3,4 fault
SCENARIOS = {
    # 1
    "faulty": [(
        simulate_choose_one, {
            "options": [
                (simulate_drift_fault, {"intensity":0.02}),
                (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                (simulate_bias_fault, {"bias":0.08}),
                (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05}),
            ]
        }
    )],

    # 2
    "drift": [(simulate_drift_fault, {"intensity":0.02})],
    "spike": [(simulate_spike_fault, {"intensity":0.08, "p":0.015})],
    "bias": [(simulate_bias_fault, {"bias":0.08})],
    "hardware": [(simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],

    # 3
    "bias+malfunc": [(simulate_bias_fault, {"bias":0.08}),
                     (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "spike+malfunc": [(simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                      (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "spike+bias": [(simulate_spike_fault, {"intensity":0.08, "p":0.015}),
                   (simulate_bias_fault, {"bias":0.08})],
    "drift+malfunc": [(simulate_drift_fault, {"intensity":0.02}),
                      (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})],
    "drift+bias": [(simulate_drift_fault, {"intensity":0.02}),
                   (simulate_bias_fault, {"bias":0.08})],
    "drift+spike": [(simulate_drift_fault, {"intensity":0.02}),
                    (simulate_spike_fault, {"intensity":0.08, "p":0.015})],

    # 4
    "spike+bias+malfunc": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "drift+bias+malfunc": [
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "spike+drift+malfunc": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05})
    ],
    "drift+spike+bias": [
        (simulate_drift_fault, {"intensity":0.02}),
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
    ],

    # 5
    "spike+bias+malfunc+drift": [
        (simulate_spike_fault, {"intensity":0.08, "p":0.015}),
        (simulate_bias_fault, {"bias":0.08}),
        (simulate_hardware_fault, {"stuck_prob":0.08, "loss_prob":0.05}),
        (simulate_drift_fault, {"intensity":0.02}),
    ],
}


In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

def make_windows(X, win, stride):
    Xn = np.asarray(X, dtype=np.float32)
    N = Xn.shape[0]
    if win <= 0 or stride <= 0:
        raise ValueError("win dan stride harus > 0")
    if N < win:
        return np.empty((0, win, Xn.shape[1]), dtype=np.float32), np.array([], dtype=int)
    view = sliding_window_view(Xn, window_shape=win, axis=0)  # (N-win+1, win, F)
    starts = np.arange(0, N - win + 1, stride, dtype=int)
    return view[starts], starts


In [ ]:

def inject_faults_multisensor(X, scenario_faults, seed=0):
    # X: (T,4) -> Y: (T,4), M: (T,4) boolean mask
    rng = np.random.default_rng(seed)
    Y = X.copy()
    M = np.zeros_like(Y, dtype=bool)
    for s in range(Y.shape[1]):
        y, m = simulate_multiple_faults(Y[:,s], scenario_faults, seed=int(rng.integers(1e9)))
        Y[:,s] = y
        M[:,s] = m
    # replace NaN (from malfunc loss) with forward fill then median per sensor
    Ydf = pd.DataFrame(Y)
    Ydf = Ydf.ffill().bfill().fillna(Ydf.median(numeric_only=True))
    return Ydf.to_numpy(), M

def window_fault_label(mask, win, stride, fault_ratio_thr=0.02):
    # mask: (T,4) -> per window label faulty if any sensor has >thr fraction
    T = len(mask)
    if win > T:
        return np.zeros(0, dtype=bool), np.array([], dtype=int)
    from numpy.lib.stride_tricks import sliding_window_view
    Wm = sliding_window_view(mask, window_shape=win, axis=0)[::stride]  # (Nwin, win, 4)
    ratio = Wm.mean(axis=1)  # (Nwin, 4)
    y = (ratio > fault_ratio_thr).any(axis=1)
    starts = np.arange(0, T-win+1, stride, dtype=int)
    return y, starts


# Build windowed dataset across scenarios + normal
fault_ratio_thr = 0.02
datasets = []
labels = []
scenario_names = ["normal"] + list(SCENARIOS.keys())

# simpan sinyal time-series per skenario (untuk analisis CV berbasis panjang data)
series_by_scenario = {}


# normal
W0, starts0 = make_windows(X_ds, WIN, STRIDE)
datasets.append(W0); labels.append(np.zeros(len(W0), dtype=int))
series_by_scenario["normal"] = X_ds

# scenarios
for k, (name, faults) in enumerate(SCENARIOS.items(), start=1):
    Y, M = inject_faults_multisensor(X_ds, faults, seed=100+k)
    series_by_scenario[name] = Y
    is_fault_win, starts = window_fault_label(M, WIN, STRIDE, fault_ratio_thr=fault_ratio_thr)
    Wk, _ = make_windows(Y, WIN, STRIDE)
    # label all windows as scenario k (including potentially non-faulty parts),
    # but you can also filter to only faulty windows:
    Wk = Wk[is_fault_win]
    datasets.append(Wk); labels.append(np.full(len(Wk), k, dtype=int))
    print(name, "windows:", len(Wk))

W_all = np.concatenate(datasets, axis=0)
y_all = np.concatenate(labels, axis=0)
print("Total windows:", W_all.shape, "Classes:", np.unique(y_all, return_counts=True))

# Pastikan format window = (N, WIN, 4). Jika terbalik (N,4,WIN) maka transpose.
if W_all.ndim==3 and W_all.shape[1]==4 and W_all.shape[2]==WIN:
    W_all = W_all.transpose(0,2,1)



### Optional: balanced sampling per class (reduction)
Jika data besar, batasi `max_per_class` untuk mempercepat perhitungan entropy dan ANN.

In [ ]:
def balanced_subsample(Xw, y, max_per_class=600, seed=0):
    rng=np.random.default_rng(seed)
    keep=[]
    for c in np.unique(y):
        idx=np.where(y==c)[0]
        if len(idx)>max_per_class:
            idx=rng.choice(idx, size=max_per_class, replace=False)
        keep.append(idx)
    keep=np.concatenate(keep)
    rng.shuffle(keep)
    return Xw[keep], y[keep]

def random_window_subsample(Xw, y, max_total=4000, seed=0):
    if len(Xw) <= max_total:
        return Xw, y
    rng=np.random.default_rng(seed)
    idx=rng.choice(np.arange(len(Xw)), size=max_total, replace=False)
    return Xw[idx], y[idx]

W_s, y_s = W_all, y_all

if USE_BALANCED_SUBSAMPLE:
    W_s, y_s = balanced_subsample(W_s, y_s, max_per_class=MAX_PER_CLASS, seed=RANDOM_SEED)

if USE_RANDOM_WINDOW_SAMPLE:
    W_s, y_s = random_window_subsample(W_s, y_s, max_total=MAX_WINDOWS_TOTAL, seed=RANDOM_SEED)

# Guard shape untuk menghindari IndexError 2D vs 3D
if W_s.ndim != 3:
    raise ValueError(
        f"Error: W_s harus 3D (N, WIN, S). Saat ini shape={W_s.shape}. "
        "Pastikan make_windows() dipanggil dan USE_WINDOWING=True."
    )

print("After subsample toggles:", W_s.shape, np.unique(y_s, return_counts=True))


# EDM–Fuzzy Entropy (Fitur Utama) + Metode Pembanding

**Tujuan:** untuk setiap window & setiap sensor, menghitung **vektor entropy multi-skala**
dengan beberapa metode untuk perbandingan: **EDM–Fuzzy, CMSE, FME, dan JSD–Fuzzy**.

## Langkah per skala `s` (EDM–Fuzzy)
1) **Coarse-graining** (`coarse_grain_mean`):  
   - Input: `x` (window 1D), `s`  
   - Output: `y` yang diperkasar (panjang ~ `WIN/s`) dengan rata-rata blok.

2) **Embedding** (`embed_matrix`):  
   - Input: `y`, dimensi `m`  
   - Output: matriks `V_m` bentuk `(Nemb, m)` (sliding window).

3) **Jarak Euclidean & Similarity fuzzy** (`fuzzy_phi`):  
   - Menghitung `phi_m` sebagai rata-rata similarity fuzzy:  
     
     \[
     \mu(d)=
rac{1}{1+(d/r)^2}
     \]
   - **Percepatan:** sampling `n_ref` embedding sebagai referensi (Monte Carlo),
     sehingga tidak perlu semua pasangan (mengurangi beban O(N²)).

4) **Entropy** (`edm_fuzzy_entropy_1d`):  
   - Hitung `phi_m` dan `phi_{m+1}` →  
     \[
     E(s)=\ln\left(
rac{\phi_m}{\phi_{m+1}}
ight)
     \]

**Output:** untuk satu sensor pada satu window → vektor `E` bentuk `(S,)`.

Catatan: untuk metode lain (CMSE, FME, JSD–Fuzzy), alur tetap mengikuti skala `s`
dengan definisi entropy masing-masing.


## 4) Entropy Multi-Method (EDM–Fuzzy, CMSE, FME, JSD–Fuzzy)
Implementasi ringkas mengikuti blok diagram:
- EDM–Fuzzy: seperti definisi di atas (jarak Euclidean + similarity fuzzy).
- CMSE: Composite Multiscale Sample Entropy (SampEn).
- FME: Fuzzy Multiscale Entropy (jarak Chebyshev + similarity fuzzy).
- JSD–Fuzzy: Jensen–Shannon divergence antar distribusi similarity fuzzy (m vs m+1).

**Percepatan:** gunakan sampling indeks embedding (Monte Carlo) sehingga tidak perlu semua pasangan vektor.

**Catatan:** pipeline di bawah kini menghitung semua metode **secara paralel** untuk perbandingan.


### 🔧 Peningkatan akurasi JSD–Fuzzy Entropy

Versi lama `jsd_fuzzy_entropy_1d` hanya mengeluarkan **1 fitur/skala**: nilai JSD
(divergensi *bentuk* histogram similarity fuzzy untuk embedding `m` vs `m+1`).
JSD membuang informasi **magnitude/lokasi** distribusi similarity, padahal itu
diskriminatif untuk klasifikasi fault.

Versi `rich=True` (default sekarang) mengeluarkan **4 fitur/skala**:
`[jsd, fe, mean_m, std_m]` — dengan `fe = log(mean(mu_m)/mean(mu_m1))`
(fuzzy-entropy ala EDM). Sampel similarity sudah dihitung untuk JSD, jadi
penambahan ini hampir tanpa biaya komputasi.

**Hasil (17-kelas fault, ANN MLP 128–64, rata-rata 3 split):**

| Varian JSD–Fuzzy | #fitur | akurasi |
|---|---|---|
| lama (1 fitur/skala) | 40 | ~0.341 |
| rich, n_ref=64, bins=20 | 160 | ~0.390 |
| rich, n_ref=128, bins=40 | 160 | ~0.424 |

Ablasi: menambahkan `mean_m` memberi lonjakan terbesar (`jsd`→`jsd+mean`:
0.34→0.41); `fe`+`std` menambah sedikit lagi. Untuk hasil terbaik, naikkan juga
`n_ref` (64→128) dan `jsd_bins` (20→40) pada cell konfigurasi entropy.
Set `rich=False` untuk perilaku lama.

In [ ]:
S = 10  # jumlah skala (akan dipakai untuk reshape & feature dim)
def coarse_grain_mean(x, s):
    n = (len(x)//s)*s
    if n <= 0:
        return np.array([], dtype=float)
    xs = x[:n].reshape(-1, s).mean(axis=1)
    return xs

def coarse_grain_multi(x, s):
    # Composite coarse-graining dengan offset 0..s-1 (CMSE)
    ys = []
    for k in range(s):
        n = (len(x)-k)//s
        if n <= 0:
            continue
        y = x[k:k+n*s].reshape(-1, s).mean(axis=1)
        if len(y) > 0:
            ys.append(y)
    return ys

def embed_matrix(y, m):
    # y: (L,) -> (L-m+1, m)
    L = len(y)
    if L < m:
        return np.empty((0, m), dtype=float)
    return np.lib.stride_tricks.sliding_window_view(y, m)

def fuzzy_phi(V, r, n_ref=256, seed=0):
    # V: (N,m). Approximate phi by sampling reference vectors i
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.nan
    if N > n_ref:
        ref = rng.choice(N, size=n_ref, replace=False)
    else:
        ref = np.arange(N)
    A = V[ref]
    a2 = np.sum(A*A, axis=1, keepdims=True)
    b2 = np.sum(V*V, axis=1, keepdims=True).T
    d2 = np.maximum(a2 + b2 - 2*(A @ V.T), 0.0)
    rr = r*r + 1e-24
    mu = 1.0/(1.0 + d2/rr)          # no sqrt
    mu[np.arange(len(ref)), ref] = 0.0  # no python loop
    Bi = mu.sum(axis=1) / (N-1)
    return Bi.mean()


def fuzzy_phi_cheb(V, r, n_ref=256, seed=0):
    # Similarity fuzzy dengan jarak Chebyshev (FME)
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.nan
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    A = V[ref]
    d = np.max(np.abs(A[:, None, :] - V[None, :, :]), axis=2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    for ri, i in enumerate(ref):
        mu[ri, i] = 0.0
    Bi = mu.sum(axis=1) / (N-1)
    return Bi.mean()

def sample_entropy_1d(y, m, r):
    # SampEn dasar untuk CMSE (pakai jarak Chebyshev)
    V_m  = embed_matrix(y, m)
    V_m1 = embed_matrix(y, m+1)

    def _count_similar(V):
        N = V.shape[0]
        if N < 2:
            return 0, 0
        count = 0
        total = 0
        for i in range(N-1):
            d = np.max(np.abs(V[i+1:] - V[i]), axis=1)
            count += np.sum(d <= r)
            total += (N - i - 1)
        return count, total

    c_m, t_m = _count_similar(V_m)
    c_m1, t_m1 = _count_similar(V_m1)
    if t_m == 0 or t_m1 == 0 or c_m == 0 or c_m1 == 0:
        return np.nan
    return -np.log((c_m1 / t_m1) / (c_m / t_m))

def edm_fuzzy_entropy_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0):
    # returns entropy vector [len(scales)]
    out=[]
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        phi_m  = fuzzy_phi(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        phi_m1 = fuzzy_phi(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (phi_m is None) or (phi_m1 is None) or (phi_m<=0) or (phi_m1<=0) or np.isnan(phi_m) or np.isnan(phi_m1):
            out.append(np.nan)
        else:
            out.append(np.log(phi_m/phi_m1))
    return np.array(out, dtype=float)

def cmse_1d(x, scales, m=2, r_ratio=0.2):
    # CMSE: Composite Multiscale Sample Entropy
    out = []
    for s in scales:
        ys = coarse_grain_multi(x, s)
        if not ys:
            out.append(np.nan); continue
        ent_list = []
        for y in ys:
            if len(y) < (m+2):
                continue
            r = r_ratio * np.std(y, ddof=1)
            ent_list.append(sample_entropy_1d(y, m=m, r=r))
        if len(ent_list) == 0:
            out.append(np.nan)
        else:
            out.append(np.nanmean(ent_list))
    return np.array(out, dtype=float)

def fme_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0):
    # FME: Fuzzy Multiscale Entropy (jarak Chebyshev)
    out=[]
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.append(np.nan); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        phi_m  = fuzzy_phi_cheb(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        phi_m1 = fuzzy_phi_cheb(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (phi_m is None) or (phi_m1 is None) or (phi_m<=0) or (phi_m1<=0) or np.isnan(phi_m) or np.isnan(phi_m1):
            out.append(np.nan)
        else:
            out.append(np.log(phi_m/phi_m1))
    return np.array(out, dtype=float)

def fuzzy_similarity_samples(V, r, n_ref=256, seed=0):
    # Ambil sampel nilai similarity fuzzy (untuk JSD)
    rng=np.random.default_rng(seed)
    N = V.shape[0]
    if N < 3:
        return np.array([], dtype=float)
    idx = np.arange(N)
    if N > n_ref:
        ref = rng.choice(idx, size=n_ref, replace=False)
    else:
        ref = idx
    A = V[ref]
    a2 = np.sum(A*A, axis=1, keepdims=True)
    b2 = np.sum(V*V, axis=1, keepdims=True).T
    d2 = a2 + b2 - 2*(A @ V.T)
    d2 = np.maximum(d2, 0.0)
    d = np.sqrt(d2)
    mu = 1.0/(1.0 + (d/(r+1e-12))**2)
    for ri, i in enumerate(ref):
        mu[ri, i] = np.nan
    return mu[~np.isnan(mu)].ravel()

def jsd_fuzzy_entropy_1d(x, scales, m=2, r_ratio=0.2, n_ref=256, seed=0, bins=20, rich=True):
    # JSD-Fuzzy (IMPROVED): selain JSD shape-divergence antara distribusi similarity
    # fuzzy (m vs m+1), kita tambahkan deskriptor MAGNITUDE distribusi yang sebelumnya
    # dibuang oleh JSD (yang hanya menangkap perbedaan *bentuk*).
    # Per skala -> [jsd, fe, mean_m, std_m] bila rich=True (default), atau [jsd] bila rich=False.
    #   jsd    : Jensen-Shannon divergence histogram similarity m vs m+1 (seperti versi lama)
    #   fe     : fuzzy-entropy = log(mean_mu_m / mean_mu_m1)  -> level kesamaan absolut
    #   mean_m : rata-rata similarity (lokasi distribusi)
    #   std_m  : sebaran similarity (skala distribusi)
    # Sampel similarity sudah dihitung utk JSD, jadi tambahan fitur ini hampir gratis
    # namun menaikkan akurasi klasifikasi ANN secara signifikan (~0.34 -> ~0.42 pada
    # eksperimen 17-kelas fault). Set rich=False untuk perilaku lama (1 fitur/skala).
    out=[]
    per = 4 if rich else 1
    bin_edges = np.linspace(0.0, 1.0, bins+1)
    eps = 1e-12
    for s in scales:
        y = coarse_grain_mean(x, s)
        if len(y) < (m+2):
            out.extend([np.nan]*per); continue
        r = r_ratio * np.std(y, ddof=1)
        V_m  = embed_matrix(y, m)
        V_m1 = embed_matrix(y, m+1)
        mu_m  = fuzzy_similarity_samples(V_m,  r, n_ref=n_ref, seed=seed+11*s)
        mu_m1 = fuzzy_similarity_samples(V_m1, r, n_ref=n_ref, seed=seed+17*s)
        if (len(mu_m) == 0) or (len(mu_m1) == 0):
            out.extend([np.nan]*per); continue
        p,_ = np.histogram(mu_m,  bins=bin_edges)
        q,_ = np.histogram(mu_m1, bins=bin_edges)
        p = p.astype(float); q = q.astype(float)
        if p.sum() == 0 or q.sum() == 0:
            out.extend([np.nan]*per); continue
        p /= p.sum(); q /= q.sum()
        m_ = 0.5 * (p + q)
        kl_p = np.sum(p * np.log((p + eps) / (m_ + eps)))
        kl_q = np.sum(q * np.log((q + eps) / (m_ + eps)))
        jsd = 0.5 * (kl_p + kl_q)
        if rich:
            fe = np.log((mu_m.mean() + eps) / (mu_m1.mean() + eps))
            out.extend([jsd, fe, mu_m.mean(), mu_m.std()])
        else:
            out.append(jsd)
    return np.array(out, dtype=float)

# quick sanity check on one window/sensor
scales = np.arange(1, S+1)
e = edm_fuzzy_entropy_1d(W_s[0,:,0], scales=scales, m=2, r_ratio=0.2, n_ref=256, seed=0)
e


In [ ]:
# Guard: pastikan scale tidak bikin coarse-grain kosong (menghindari 'Mean of empty slice')
m_embed = 2  # samakan dengan m pada entropy
min_len = m_embed + 2
max_scale = max(1, WIN // min_len)

# scales bisa datang dari cell sebelumnya; pastikan list int
scales = [int(s) for s in scales]
scales = [s for s in scales if s <= max_scale]
if len(scales) == 0:
    scales = [1]

# update S supaya selalu konsisten dengan scales yang benar-benar dipakai
S = len(scales)
scales = np.array(scales, dtype=int)

print("Using scales:", scales.tolist(), "S:", S, "max_scale:", max_scale)


# Fitur Akhir: Konkatenasi 4 Sensor (Multi-Method)

`compute_features_entropy(W, scales, method=...)`

**Input**
- `W`: `(Nwin, WIN, 4)` kumpulan window
- `scales`: `1..S`
- `method`: `EDM-Fuzzy`, `CMSE`, `FME`, `JSD-Fuzzy`

**Proses**
- untuk tiap window:
  - hitung entropy `(S,)` untuk sensor 1..4
  - gabungkan menjadi `(4*S,)`: `[E1, E2, E3, E4]`

**Output**
- `F_by_method`: dict fitur `(Nwin, 4*S)` untuk setiap metode
- NaN (jika ada) diimputasi dengan median per kolom.


## 5) Hitung entropy per sensor → gabung fitur
Entropy dihitung per sensor menghasilkan vector:
- Sensor1: $E_1(1..S)$
- Sensor2: $E_2(1..S)$
- Sensor3: $E_3(1..S)$
- Sensor4: $E_4(1..S)$

Fitur akhir = konkatenasi $[E_1,E_2,E_3,E_4]$ ukuran `4*S`.

Catatan A: uji beberapa pilihan skala untuk CV (kestabilan entropy).
Catatan B: semua metode dihitung **bersamaan** agar perbandingan mudah.


In [ ]:
from joblib import Parallel, delayed

START_TIME = time.time()

def time_left_sec():
    if 'KAGGLE_TIME_BUDGET_H' not in globals():
        return None
    return KAGGLE_TIME_BUDGET_H * 3600 - (time.time() - START_TIME)

def ensure_window_3d(W, name="W"):
    if W.ndim != 3:
        raise ValueError(f"{name} harus 3D (N, WIN, S). Dapat {W.shape}.")
    return W

def sanitize_features(F, name="F"):
    Fdf = pd.DataFrame(F)
    if Fdf.isna().any().any():
        warnings.warn(f"{name} masih ada NaN; imputasi median diterapkan.")
        Fdf = Fdf.fillna(Fdf.median(numeric_only=True))
    if Fdf.isna().any().any():
        raise ValueError(f"Error-nya jelas: fitur {name} masih ada NaN setelah imputasi.")
    return Fdf.to_numpy()

def compute_features_entropy(W, scales, method=None, m=2, r_ratio=0.2, n_ref=256, jsd_bins=20, seed=0, n_jobs=-1, prefer="processes"):
    # W: (Nwin, win, 4) -> (Nwin, 4*len(scales))
    W = ensure_window_3d(W, name="W")
    Nwin, win, ns = W.shape
    if method is None:
        method = DEFAULT_METHOD
    method_key = str(method).strip().lower()

    def entropy_1d(x, seed_local):
        if method_key == 'edm-fuzzy':
            return edm_fuzzy_entropy_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local)
        if method_key == 'cmse':
            return cmse_1d(x, scales=scales, m=m, r_ratio=r_ratio)
        if method_key == 'fme':
            return fme_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local)
        if method_key == 'jsd-fuzzy':
            return jsd_fuzzy_entropy_1d(x, scales=scales, m=m, r_ratio=r_ratio, n_ref=n_ref, seed=seed_local, bins=jsd_bins)
        raise ValueError("Unknown method: %s" % method)

    def one_window(i):
        feats = [entropy_1d(W[i,:,s], seed_local=seed+1000*i+19*s) for s in range(ns)]
        return np.concatenate(feats, axis=0)

    if n_jobs == 1 or Nwin <= 1:
        F = np.vstack([one_window(i) for i in range(Nwin)])
    else:
        F = Parallel(n_jobs=n_jobs, prefer=prefer)(delayed(one_window)(i) for i in range(Nwin))
        F = np.vstack(F)
    return F


# Utility logging untuk footprint komputasi
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

def run_with_metrics(label, fn):
    # Catatan: footprint = estimasi peak memory Python via tracemalloc
    tracemalloc.start()
    t0 = time.perf_counter()
    c0 = time.process_time()
    result = fn()
    t1 = time.perf_counter()
    c1 = time.process_time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    metrics = {
        "wall_s": t1 - t0,
        "cpu_s": c1 - c0,
        "peak_mem_mb": peak / (1024*1024)
    }
    logging.info("%s | wall=%.2fs cpu=%.2fs peak_mem=%.2f MB", label, metrics["wall_s"], metrics["cpu_s"], metrics["peak_mem_mb"])
    return result, metrics

# === Entropy params (speed) ===
# S kecil + n_ref kecil = jauh lebih cepat, masih cukup untuk baseline eksperimen.
S = 10
scales = np.arange(1, S+1)
m = 2
r_ratio = 0.2
n_ref = 128
jsd_bins = 40

# Default: fokus ke EDM-Fuzzy dulu (metode lain bisa ditambah lagi kalau perlu)
methods = METHOD_LIST if RUN_ALL_METHODS else [DEFAULT_METHOD]
F_by_method = {}
F_metrics = {}

# PENTING: cache key HARUS mencakup konfigurasi fitur (jumlah skala, n_ref,
# jsd_bins, rich). Bug sebelumnya: cache hanya di-key oleh nama metode, jadi
# saat config/kode diubah (mis. JSD versi 'rich') fitur lama yang basi tetap
# dimuat dan tak pernah dihitung ulang -> JSD muncul 24 fitur (versi lama
# rich=False) alih-alih 4 fitur/skala. Tag config bikin cache invalid otomatis
# begitu salah satu parameter berubah.
JSD_RICH = True
CFG_TAG = f"S{len(scales)}_m{m}_r{r_ratio}_nref{n_ref}_bins{jsd_bins}_rich{int(JSD_RICH)}"

for name in methods:
    safe = name.replace("-", "_").replace(" ", "_")
    cache_path = os.path.join(CACHE_DIR, f"F_{safe}__{CFG_TAG}.npy")
    if CACHE_FEATURES and os.path.exists(cache_path):
        F_by_method[name] = sanitize_features(np.load(cache_path), name=f"F_{name}")
        print(name, "feature shape (cache):", F_by_method[name].shape, "| cfg:", CFG_TAG)
        continue
    Fm, mtr = run_with_metrics(
        "Entropy %s" % name,
        lambda n=name: compute_features_entropy(W_s, scales=scales, method=n, m=m, r_ratio=r_ratio, n_ref=n_ref, jsd_bins=jsd_bins, seed=7)
    )
    F_by_method[name] = sanitize_features(Fm, name=f"F_{name}")
    F_metrics[name] = mtr
    print(name, "feature shape:", F_by_method[name].shape, "| cfg:", CFG_TAG)
    # simpan cache per-metode DENGAN tag config, di dalam loop -> semua metode tersimpan
    # (sebelumnya hanya metode terakhir yang ter-cache karena di luar loop).
    if CACHE_FEATURES:
        np.save(cache_path, F_by_method[name])

pd.DataFrame(F_metrics).T

# Default: gunakan EDM–Fuzzy agar blok berikut tetap kompatibel
F = F_by_method[DEFAULT_METHOD]
if CACHE_FEATURES:
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(os.path.join(CACHE_DIR, "y_windows.npy"), y_s)
    np.save(os.path.join(CACHE_DIR, "F_default.npy"), F)
    print("Saved per-method caches (cfg:", CFG_TAG, ") + y_windows.npy")


> **Catatan eksekusi:** Blok legacy di bawah ini (CV-stabilitas vs panjang data mentah 2000/7000/10000, tren akurasi vs S, dan tabel trade-off ANN per skenario) **sengaja tidak dieksekusi ulang** pada run ini -- cell CV-vs-panjang-data menguji 17 skenario x 3 panjang x 4 metode x 30 pengulangan tanpa windowing, yang di run interaktif sebelumnya menghabiskan beberapa jam wall-clock (lihat jejak jam eksekusi lama: 08:15 -> 15:45 -> 20:02 pada tanggal yang sama di riwayat notebook ini). Blok ini independen dari fitur entropy di atasnya dan tidak dipakai oleh evaluasi RQ1-RQ5 di bawah, sehingga tidak memengaruhi hasil perbandingan EDM-Fuzzy vs JSD-Fuzzy. Jalankan cell ini manual/terpisah bila memang perlu tabel/plotnya.

In [ ]:
# Alias kompatibilitas (beberapa blok lama masih pakai X_feat/y)
X_feat = F
y = y_s
print('Alias set: X_feat=F', getattr(X_feat,'shape',None), 'y', getattr(y,'shape',None))


In [ ]:
# === Definisi Lima Skenario Klasifikasi Paper (S1–S5) ===
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import warnings

# Pastikan variabel global tersedia
assert "y_s" in globals(), "Jalankan cell pembentukan dataset (fault injection + balanced subsample) terlebih dahulu."
assert "F_by_method" in globals() and len(F_by_method) >= 2, "Jalankan cell komputasi fitur entropy terlebih dahulu."

METHODS_PAPER = ["EDM-Fuzzy", "JSD-Fuzzy"]

# --- Bangun label per skenario ---
# Scenario 1 (S1): binary – normal (0) vs faulty (1)
def build_labels_S1(y_full):
    return (y_full > 0).astype(int)

# Scenario 2 (S2): fault-free vs 4 single-fault types (5-class)
# single-fault indices di y_s: lihat scenario_names
SINGLE_FAULT_NAMES = ["drift", "spike", "bias", "hardware"]
def build_labels_S2(y_full, scenario_names):
    sf_idx = [scenario_names.index(n) for n in SINGLE_FAULT_NAMES if n in scenario_names]
    keep = np.where((y_full == 0) | np.isin(y_full, sf_idx))[0]
    y_raw = y_full[keep]
    # Remap: 0 tetap 0, tiap sf_idx -> 1,2,3,4
    remap = {0: 0}
    for new_label, old_idx in enumerate(sf_idx, start=1):
        remap[old_idx] = new_label
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

# Scenario 3 (S3): fault-free vs 2-fault combinations (multi kelas)
MULTI2_FAULT_NAMES = ["bias+malfunc", "spike+malfunc", "spike+bias",
                       "drift+malfunc", "drift+bias", "drift+spike"]
def build_labels_S3(y_full, scenario_names):
    mf2_idx = [scenario_names.index(n) for n in MULTI2_FAULT_NAMES if n in scenario_names]
    keep = np.where((y_full == 0) | np.isin(y_full, mf2_idx))[0]
    y_raw = y_full[keep]
    remap = {0: 0}
    for new_label, old_idx in enumerate(mf2_idx, start=1):
        remap[old_idx] = new_label
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

# Scenario 4 (S4): single-fault identification (4-class, tanpa normal)
def build_labels_S4(y_full, scenario_names):
    sf_idx = [scenario_names.index(n) for n in SINGLE_FAULT_NAMES if n in scenario_names]
    keep = np.where(np.isin(y_full, sf_idx))[0]
    y_raw = y_full[keep]
    remap = {old: new for new, old in enumerate(sf_idx)}
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

# Scenario 5 (S5): all-class – normal + semua single-fault + 2-fault combinations
def build_labels_S5(y_full, scenario_names):
    all_named = SINGLE_FAULT_NAMES + MULTI2_FAULT_NAMES
    all_idx = [scenario_names.index(n) for n in all_named if n in scenario_names]
    keep = np.where((y_full == 0) | np.isin(y_full, all_idx))[0]
    y_raw = y_full[keep]
    all_classes = sorted(set([0] + all_idx))
    remap = {old: new for new, old in enumerate(all_classes)}
    y_new = np.array([remap[v] for v in y_raw])
    return keep, y_new

print("Skenario S1–S5 siap dibentuk.")
print("scenario_names:", scenario_names[:10], "...")


# Revisi: Per-Skenario + Fitur Time-Domain + XGBoost

**Catatan revisi (2026-07-03).**

Motivasi: akurasi 17-kelas rendah (~0.41) karena (a) kelas terlalu banyak & saling overlap, (b) fitur entropy hanya ringkasan skalar, (c) data terbatas (~117 window/kelas).

Revisi yang dicoba di bawah:
1. **Klasifikasi per-skenario (S1-S5)** dengan **grid search terpisah tiap skenario** (bukan satu model 17-kelas). Kelas lebih sedikit -> lebih mudah.
2. **Fitur time-domain** (mean, std, rms, min, max, range, median, MAD, skew, kurtosis, zero-crossing, slope, energy, 3 band energi FFT) per sensor per window, **digabung** dengan fitur entropy.
3. **XGBoost** (gradient boosting) menggantikan MLP - biasanya lebih kuat untuk fitur tabular berdimensi kecil.
4. **Ablasi feature-set**: EDM, JSD, Time, EDM+Time, JSD+Time -> lihat kombinasi terbaik per skenario.

Catatan Kaggle:
- `METHOD_LIST` dipangkas ke **EDM + JSD saja** (skip CMSE/FME yang lambat, terutama CMSE exact O(N^2)) supaya feature-extraction cepat.
- **Protokol fair dijaga**: grid search hanya di **train set** (inner 3-fold CV), test set tetap held-out, `random_state=42` sama untuk semua metode & feature-set.
- Grid XGBoost sengaja kecil supaya total runtime wajar di Kaggle CPU (~15-25 menit).


In [ ]:
# === Fitur Time-Domain per window per sensor (tambahan revisi) ===
import numpy as np
from scipy import stats as _sstats

def time_domain_features(W):
    """W: (N, T, S) -> (N, S*nblok). Fitur statistik + FFT band per sensor."""
    W = np.asarray(W, dtype=float)
    N, T, S = W.shape
    t = np.arange(T); tc = t - t.mean(); tden = (tc**2).sum() or 1.0
    mean = W.mean(1); std = W.std(1)
    rms  = np.sqrt((W**2).mean(1))
    mn   = W.min(1); mx = W.max(1); ptp = mx - mn
    med  = np.median(W, 1)
    mad  = np.median(np.abs(W - med[:, None, :]), 1)
    sk   = _sstats.skew(W, axis=1)
    ku   = _sstats.kurtosis(W, axis=1)
    z    = W - mean[:, None, :]
    zcr  = (np.diff(np.sign(z), axis=1) != 0).sum(1) / max(1, (T - 1))
    slope= (z * tc[None, :, None]).sum(1) / tden
    energy = (W**2).sum(1)
    mag = np.abs(np.fft.rfft(z, axis=1))            # (N, F, S)
    Fb = mag.shape[1]; b = max(1, Fb // 3)
    fb1 = (mag[:, :b, :]**2).sum(1)
    fb2 = (mag[:, b:2*b, :]**2).sum(1)
    fb3 = (mag[:, 2*b:, :]**2).sum(1)
    blocks = [mean, std, rms, mn, mx, ptp, med, mad, sk, ku, zcr, slope, energy, fb1, fb2, fb3]
    Tf = np.concatenate(blocks, axis=1)             # (N, S*16)
    return np.nan_to_num(Tf, nan=0.0, posinf=0.0, neginf=0.0)

T_time = time_domain_features(W_s)
print("Time-domain feature shape:", T_time.shape, "(", T_time.shape[1]//W_s.shape[2], "fitur x", W_s.shape[2], "sensor )")


In [ ]:
# === Per-Skenario: XGBoost + ablasi feature-set (grid search per skenario) ===
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False
    print("xgboost tak tersedia, fallback HistGradientBoosting:", e)

# Feature-set untuk ablasi
FE = {"EDM": F_by_method["EDM-Fuzzy"], "JSD": F_by_method["JSD-Fuzzy"], "Time": T_time}
FE["EDM+Time"] = np.hstack([FE["EDM"], FE["Time"]])
FE["JSD+Time"] = np.hstack([FE["JSD"], FE["Time"]])
FEATURE_SETS = ["EDM", "JSD", "Time", "EDM+Time", "JSD+Time"]

def make_clf():
    if HAS_XGB:
        base = XGBClassifier(tree_method="hist", n_jobs=1, random_state=42, verbosity=0)
        grid = {"clf__n_estimators": [300, 600], "clf__max_depth": [4, 6],
                "clf__learning_rate": [0.05, 0.1]}
    else:
        base = HistGradientBoostingClassifier(random_state=42)
        grid = {"clf__max_depth": [None, 6], "clf__learning_rate": [0.05, 0.1]}
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("clf", base)]), grid

def build_scen(y_full, names):
    d = {"S1": (np.arange(len(y_full)), build_labels_S1(y_full))}
    for nm, fn in [("S2", build_labels_S2), ("S3", build_labels_S3),
                   ("S4", build_labels_S4), ("S5", build_labels_S5)]:
        keep, yn = fn(y_full, names); d[nm] = (keep, yn)
    return d

SEED = 42
scen = build_scen(y_s, scenario_names)
rows = []
scen_store = {}  # (sname, fs) -> {yte, pred, class_names, gs} for CM + loss inspection below

def scenario_class_names(sname, classes):
    if sname == "S1":
        return ["fault-free", "faulty"]
    return [scenario_names[c] for c in classes]

for sname, (keep, yscen) in scen.items():
    classes = np.unique(yscen); remap = {c: i for i, c in enumerate(classes)}
    yy = np.array([remap[v] for v in yscen])
    ncls = len(classes)
    class_names_s = scenario_class_names(sname, classes)
    for fs in FEATURE_SETS:
        X = FE[fs][keep]
        Xtr, Xte, ytr, yte = train_test_split(X, yy, test_size=0.25,
                                              random_state=SEED, stratify=yy)
        pipe, grid = make_clf()
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
        gs = GridSearchCV(pipe, grid, cv=cv, scoring="f1_macro", n_jobs=-1)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            gs.fit(Xtr, ytr)
        pred = gs.predict(Xte)
        rows.append({
            "Scenario": sname, "n_classes": ncls, "FeatureSet": fs, "n_feat": X.shape[1],
            "Accuracy": round(accuracy_score(yte, pred), 4),
            "Precision": round(precision_score(yte, pred, average="macro", zero_division=0), 4),
            "Recall": round(recall_score(yte, pred, average="macro", zero_division=0), 4),
            "F1_macro": round(f1_score(yte, pred, average="macro", zero_division=0), 4),
            "CV_F1": round(gs.best_score_, 4),
        })
        proba = gs.predict_proba(Xte)
        scen_store[(sname, fs)] = {"yte": yte, "pred": pred, "proba": proba, "class_names": class_names_s, "gs": gs, "n_classes": ncls}
        print(f"{sname:3} {fs:9} feat={X.shape[1]:4} acc={rows[-1]['Accuracy']:.3f} F1={rows[-1]['F1_macro']:.3f}")

results_xgb = pd.DataFrame(rows)
print("\nSelesai:", len(results_xgb), "kombinasi (skenario x feature-set).")


## Confusion Matrix - Semua Kombinasi (5 Skenario x 5 Feature-Set = 25 CM)

Setiap kombinasi skenario (S1-S5) x feature-set (EDM, JSD, Time, EDM+Time, JSD+Time) punya model
XGBoost terbaik (hasil GridSearch) tersendiri, disimpan di `scen_store`. Sel di bawah menampilkan
confusion matrix + classification report untuk semua 25 kombinasi, memakai split test yang sama
(seed=42) yang dipakai untuk menghitung tabel `results_xgb`.

In [ ]:
for (sname, fs), st in scen_store.items():
    yte, pred, class_names_s = st["yte"], st["pred"], st["class_names"]
    print(f"=== Confusion Matrix - {sname} | {fs} ===")
    print(classification_report(yte, pred, labels=np.arange(len(class_names_s)),
                                  target_names=class_names_s, zero_division=0))
    acc = accuracy_score(yte, pred); f1m = f1_score(yte, pred, average="macro")
    print(f"Accuracy: {acc:.4f}  Macro-F1: {f1m:.4f}")
    cm = confusion_matrix(yte, pred, labels=np.arange(len(class_names_s)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names_s)
    fig, ax = plt.subplots(figsize=(max(5, 0.9*len(class_names_s)), max(4.5, 0.8*len(class_names_s))))
    disp.plot(xticks_rotation=45, ax=ax, colorbar=False)
    ax.set_title(f"CM - {sname} ({fs})")
    plt.tight_layout()
    plt.show()

## Loss Function - Bagaimana Loss Dihitung (XGBoost)

Classifier revisi ini adalah `XGBClassifier` (`tree_method="hist"`). Untuk klasifikasi multi-kelas,
objective default XGBoost adalah **`multi:softprob`**, dioptimasi memakai **multiclass log-loss
(mlogloss)** di tiap boosting round (gradient boosting menambah satu tree baru per round untuk
mengurangi loss ini):

$$L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K} y_{ik} \log(\hat{p}_{ik}) \; + \; \sum_{t} \Omega(f_t)$$

- $\hat{p}_{ik}$ = probabilitas softmax dari jumlah skor semua tree ke-$t$ untuk kelas $k$.
- $\Omega(f_t)$ = regularisasi kompleksitas tree ke-$t$ (jumlah daun + L2 pada bobot daun).
- Untuk skenario **S1 (binary)**, XGBoost otomatis pakai objective `binary:logistic` (loss =
  binary log-loss / cross-entropy 2 kelas) - rumus sama, $K=2$.

Bedanya dengan MLP (2 notebook lain): MLP hitung loss lewat backpropagation di jaringan neuron;
XGBoost hitung loss lewat gradient boosting - tiap tree baru dilatih untuk mengoreksi residual
gradien loss dari tree-tree sebelumnya. Sel di bawah menghitung ulang mlogloss/logloss di test set
untuk model terbaik tiap kombinasi (nilai loss tidak disimpan otomatis oleh GridSearchCV, jadi
dihitung eksplisit dari predicted probability).

In [ ]:
from sklearn.metrics import log_loss

loss_rows = []
for (sname, fs), st in scen_store.items():
    yte, proba = st["yte"], st["proba"]
    ll = log_loss(yte, proba, labels=np.arange(st["n_classes"]))
    best_pipe = st["gs"].best_estimator_
    clf_params = best_pipe.named_steps["clf"].get_params()
    loss_rows.append({
        "Scenario": sname, "FeatureSet": fs,
        "test_logloss": round(ll, 4),
        "n_estimators": clf_params.get("n_estimators"),
        "max_depth": clf_params.get("max_depth"),
        "learning_rate": clf_params.get("learning_rate"),
    })

loss_table = pd.DataFrame(loss_rows)
loss_table

In [ ]:
# === Ringkasan + simpan CSV ===
import os
os.makedirs("exports", exist_ok=True)

print("=== XGBoost per-skenario x feature-set (metrik di TEST set, split=42) ===")
print(results_xgb.to_string(index=False))

# feature-set terbaik per skenario (by F1_macro)
best = (results_xgb.sort_values("F1_macro")
        .groupby("Scenario", as_index=False).tail(1)
        .sort_values("Scenario"))
print("\n=== Feature-set TERBAIK per skenario ===")
print(best[["Scenario", "n_classes", "FeatureSet", "n_feat", "Accuracy", "F1_macro"]].to_string(index=False))

# perbandingan vs entropy-only (JSD) sebagai baseline revisi
piv = results_xgb.pivot(index="Scenario", columns="FeatureSet", values="F1_macro")
print("\n=== F1_macro per feature-set (baris=skenario) ===")
print(piv.to_string())

results_xgb.to_csv("exports/xgb_perscenario_results.csv", index=False)
best.to_csv("exports/xgb_perscenario_best.csv", index=False)
print("\n[saved] exports/xgb_perscenario_results.csv")
print("[saved] exports/xgb_perscenario_best.csv")
print("\nCATATAN: accuracy S1 tinggi karena imbalance (94% fault) -> baca F1_macro. "
      "Klaim akurasi pakai skenario balanced (S2-S5) + feature-set terbaik.")
